# Diurnal Multi-Wavelength Analysis — Publication Figures

Generates slide-ready figures for the supplementary deck.
Each figure maps to a specific slide placeholder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import os, warnings
warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 8,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
})

OUT = 'output/plots/diurnal_pub'
os.makedirs(OUT, exist_ok=True)

# Wavelength config
WL = {
    'UV':    {'nm': 375, 'color': '#7B2FBE'},
    'Blue':  {'nm': 470, 'color': '#2196F3'},
    'Green': {'nm': 528, 'color': '#4CAF50'},
    'Red':   {'nm': 625, 'color': '#F44336'},
    'IR':    {'nm': 880, 'color': '#212121'},
}
BC = {name: f'{name} BCc' for name in WL}
BC_SM = {name: f'{name} BCc smoothed' for name in WL}

SEASONS = {
    'Dry (Oct-Feb)': [10, 11, 12, 1, 2],
    'Belg (Mar-May)': [3, 4, 5],
    'Kiremt (Jun-Sep)': [6, 7, 8, 9],
}
SCOL = {'Dry (Oct-Feb)': '#E67E22', 'Belg (Mar-May)': '#27AE60', 'Kiremt (Jun-Sep)': '#3498DB'}

def month_to_season(m):
    for s, months in SEASONS.items():
        if m in months:
            return s

print('Setup complete')

In [ ]:
# Load and QC the 1-minute MA350 data
DATA_PATH = (
    "/Users/ahmadjalil/Desktop/"
    "JacrosMA350 60s Data20250804082112/"
    "df_jacros_cleaned_API_and_OG_manual_BC_all_wl.pkl"
)
df_raw = pd.read_pickle(DATA_PATH)
df_raw['datetime_local'] = pd.to_datetime(df_raw['datetime_local'])
df_raw = df_raw.set_index('datetime_local').sort_index()

all_bc = list(BC.values()) + list(BC_SM.values())
existing_bc = [c for c in all_bc if c in df_raw.columns]
df = df_raw[existing_bc].copy()
df[existing_bc] = df[existing_bc] / 1000.0  # ng -> ug

for c in existing_bc:
    df.loc[df[c] < 0, c] = np.nan
for c in existing_bc:
    valid = df[c].dropna()
    df.loc[df[c] > valid.mean() + 3 * valid.std(), c] = np.nan

df['Hour'] = df.index.hour
df['Month'] = df.index.month
df['Date'] = df.index.date
df['Season'] = df['Month'].map(month_to_season)

# Derived metrics
ir_col = BC['IR']
ratio_cols = {}
for name in ['UV', 'Blue', 'Green', 'Red']:
    col = BC[name]
    rcol = f'{name}/IR ratio'
    mask = (df[col].notna()) & (df[ir_col].notna()) & (df[ir_col] > 0.1)
    df[rcol] = np.where(mask, df[col] / df[ir_col], np.nan)
    ratio_cols[name] = rcol

# TRUE AAE (= apparent + 1, because MA350 applies MAC with AAE=1 for calibration)
uv_col = BC['UV']
wl_uv, wl_ir = WL['UV']['nm'], WL['IR']['nm']
mask = (df[uv_col] > 0.1) & (df[ir_col] > 0.1)
df['AAE_apparent'] = np.where(mask, -np.log(df[uv_col] / df[ir_col]) / np.log(wl_uv / wl_ir), np.nan)
df['AAE_true'] = df['AAE_apparent'] + 1.0  # correct for MA350 MAC calibration

blue_col = BC['Blue']
wl_blue = WL['Blue']['nm']
mask2 = (df[blue_col] > 0.1) & (df[ir_col] > 0.1)
df['AAE_Blue_apparent'] = np.where(mask2, -np.log(df[blue_col] / df[ir_col]) / np.log(wl_blue / wl_ir), np.nan)
df['AAE_Blue_true'] = df['AAE_Blue_apparent'] + 1.0

df['Delta_C'] = df[uv_col] - df[ir_col]

bc_matrix = df[list(BC.values())]
row_mean = bc_matrix.mean(axis=1)
row_std = bc_matrix.std(axis=1)
df['wl_CV'] = (row_std / row_mean).where(row_mean > 0.1)

print(f'Loaded {len(df):,} records, {df[list(BC.values())].notna().all(axis=1).sum():,} with all 5 wavelengths')

## Slide 5: Multi-Wavelength Diurnal Overlay

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (bc_set, label) in enumerate([(BC, 'Raw BCc'), (BC_SM, 'Smoothed BCc')]):
    ax = axes[ax_idx]
    for name, info in WL.items():
        col = bc_set[name]
        if col not in df.columns:
            continue
        hourly = df.groupby('Hour')[col]
        med = hourly.median()
        q25, q75 = hourly.quantile(0.25), hourly.quantile(0.75)
        ax.plot(med.index, med.values, marker='o', ms=4, lw=2,
                color=info['color'], label=f"{name} ({info['nm']} nm)")
        ax.fill_between(med.index, q25, q75, alpha=0.12, color=info['color'])
    ax.set_xlabel('Hour of Day (local)')
    ax.set_ylabel('BC concentration (ug/m3)')
    ax.set_title(f'Median Diurnal BC by Wavelength ({label})\n(shading = IQR)')
    ax.set_xticks(range(0, 24, 2))
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUT}/slide05_diurnal_all_wavelengths.png')
plt.show()

## Slide 6: Wavelength / IR Ratios (Spectral Shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: all 4 ratios
ax = axes[0]
for name in ['UV', 'Blue', 'Green', 'Red']:
    rcol = ratio_cols[name]
    info = WL[name]
    med = df.groupby('Hour')[rcol].median()
    q25, q75 = df.groupby('Hour')[rcol].quantile(0.25), df.groupby('Hour')[rcol].quantile(0.75)
    ax.plot(med.index, med.values, marker='o', ms=4, lw=2,
            color=info['color'], label=f"{name}/IR ({info['nm']}/{WL['IR']['nm']} nm)")
    ax.fill_between(med.index, q25, q75, alpha=0.12, color=info['color'])
ax.axhline(1.0, color='gray', ls='--', lw=1, label='IR = IR (1.0)')
ax.set_xlabel('Hour of Day (local)')
ax.set_ylabel('BC wavelength / IR BCc ratio')
ax.set_title('Diurnal Spectral Shape\n(>1 = enhanced short-wave absorption)')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=8)

# Right: UV/IR zoom
ax = axes[1]
rcol = ratio_cols['UV']
hourly = df.groupby('Hour')[rcol]
med = hourly.median()
q25, q75 = hourly.quantile(0.25), hourly.quantile(0.75)
ax.plot(med.index, med.values, 'o-', lw=2.5, color=WL['UV']['color'], label='UV/IR median')
ax.fill_between(med.index, q25, q75, alpha=0.15, color=WL['UV']['color'])
ax.axhline(1.0, color='gray', ls='--', lw=1)

peak_h, min_h = med.idxmax(), med.idxmin()
ax.annotate(f'Peak: {peak_h:02d}:00\n({med[peak_h]:.2f})', xy=(peak_h, med[peak_h]),
            xytext=(peak_h+2, med[peak_h]+0.05), fontsize=9, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=WL['UV']['color']))
ax.annotate(f'Min: {min_h:02d}:00\n({med[min_h]:.2f})', xy=(min_h, med[min_h]),
            xytext=(min_h+2, med[min_h]-0.05), fontsize=9, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=WL['UV']['color']))

ax.set_xlabel('Hour of Day (local)')
ax.set_ylabel('UV BCc / IR BCc')
ax.set_title('UV/IR Ratio Diurnal Pattern\n(>1 = biomass/brown carbon signal)')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUT}/slide06_wavelength_ratios.png')
plt.show()

print(f'UV/IR ratio: peak at {peak_h:02d}:00 ({med[peak_h]:.2f}), min at {min_h:02d}:00 ({med[min_h]:.2f}), range={med.max()-med.min():.2f}')

## Slide 7: AAE Diurnal (TRUE AAE = apparent + 1)

In [ ]:
time_blocks = {
    '00-05\n(night)':     (0, 5),
    '05-09\n(morning)':   (5, 9),
    '09-13\n(midday)':    (9, 13),
    '13-17\n(afternoon)': (13, 17),
    '17-21\n(evening)':   (17, 21),
    '21-24\n(late night)':(21, 24),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: TRUE AAE diurnal curves
ax = axes[0]
for aae_col, color, label in [
    ('AAE_true', WL['UV']['color'], 'AAE (UV/IR, 375/880 nm)'),
    ('AAE_Blue_true', WL['Blue']['color'], 'AAE (Blue/IR, 470/880 nm)'),
]:
    hourly = df.groupby('Hour')[aae_col]
    med = hourly.median()
    q25, q75 = hourly.quantile(0.25), hourly.quantile(0.75)
    ax.plot(med.index, med.values, 'o-', lw=2, ms=4, color=color, label=label)
    ax.fill_between(med.index, q25, q75, alpha=0.12, color=color)

ax.axhline(1.0, color='#E74C3C', ls='--', lw=1.5, alpha=0.7, label='AAE=1.0 (fossil fuel)')
ax.axhline(2.0, color='#8B4513', ls='--', lw=1.5, alpha=0.7, label='AAE=2.0 (biomass)')
ax.set_xlabel('Hour of Day (local)')
ax.set_ylabel('Absorption Angstrom Exponent (true)')
ax.set_title('True AAE Diurnal Pattern\n(MA350 apparent AAE + 1.0 correction)')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=8)

# Right: TRUE AAE by time block
ax = axes[1]
block_data, block_labels = [], []
block_colors = ['#1a237e', '#ff8f00', '#e65100', '#bf360c', '#4a148c', '#263238']
for label, (h0, h1) in time_blocks.items():
    mask = (df['Hour'] >= h0) & (df['Hour'] < h1)
    block_data.append(df.loc[mask, 'AAE_true'].dropna())
    block_labels.append(label)

bp = ax.boxplot(block_data, tick_labels=block_labels, patch_artist=True,
                showfliers=False, whis=[10, 90])
for patch, color in zip(bp['boxes'], block_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.axhline(1.0, color='#E74C3C', ls='--', lw=1.5, alpha=0.7)
ax.axhline(2.0, color='#8B4513', ls='--', lw=1.5, alpha=0.7)
ax.set_ylabel('True AAE (UV/IR)')
ax.set_title('AAE Distribution by Time Block\n(all > 1.0 = persistent biomass/brown carbon)')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{OUT}/slide07_diurnal_AAE.png')
plt.show()

print("True AAE (UV/IR) by time block (median):")
for label, (h0, h1) in time_blocks.items():
    mask = (df['Hour'] >= h0) & (df['Hour'] < h1)
    med = df.loc[mask, 'AAE_true'].median()
    print(f"  {label.replace(chr(10),' ')}: {med:.2f}")

In [ ]:
# Slide 8: Delta-C Diurnal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
hourly = df.groupby('Hour')['Delta_C']
med = hourly.median()
q25, q75 = hourly.quantile(0.25), hourly.quantile(0.75)
ax.plot(med.index, med.values, 'o-', lw=2.5, ms=5, color='#8B4513')
ax.fill_between(med.index, q25, q75, alpha=0.2, color='#8B4513')
ax.axhline(0, color='gray', ls='--', lw=1)
ax.set_xlabel('Hour of Day (local)')
ax.set_ylabel('Delta-C (UV - IR BCc, ug/m3)')
ax.set_title('Delta-C Diurnal Pattern\n(positive = biomass burning signal)')
ax.set_xticks(range(0, 24, 2))

ax = axes[1]
for season, color in SCOL.items():
    sdata = df[df['Season'] == season]
    med = sdata.groupby('Hour')['Delta_C'].median()
    ax.plot(med.index, med.values, 'o-', lw=2, ms=4, color=color, label=season)
ax.axhline(0, color='gray', ls='--', lw=1)
ax.set_xlabel('Hour of Day (local)')
ax.set_ylabel('Delta-C (UV - IR BCc, ug/m3)')
ax.set_title('Delta-C Diurnal Pattern by Season')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUT}/slide08_diurnal_deltaC.png')
plt.show()

In [ ]:
# Slide 9: Wavelength Spread CV
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
hourly = df.groupby('Hour')['wl_CV']
med = hourly.median()
q25, q75 = hourly.quantile(0.25), hourly.quantile(0.75)
ax.plot(med.index, med.values, 'o-', lw=2.5, ms=5, color='#1565C0')
ax.fill_between(med.index, q25, q75, alpha=0.15, color='#1565C0')
ax.set_xlabel('Hour of Day (local)')
ax.set_ylabel('CV across 5 wavelengths')
ax.set_title('Wavelength Spread (CV) by Hour\n(higher = wavelengths more different)')
ax.set_xticks(range(0, 24, 2))

ax = axes[1]
for season, color in SCOL.items():
    sdata = df[df['Season'] == season]
    med = sdata.groupby('Hour')['wl_CV'].median()
    ax.plot(med.index, med.values, 'o-', lw=2, ms=4, color=color, label=season)
ax.set_xlabel('Hour of Day (local)')
ax.set_ylabel('CV across 5 wavelengths')
ax.set_title('Wavelength Spread by Hour and Season')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUT}/slide09_wavelength_spread.png')
plt.show()

In [ ]:
# Slide 10: Seasonal Diurnal Multi-Wavelength (2x3 panel)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
season_list = list(SEASONS.keys())

for idx, season in enumerate(season_list):
    sdata = df[df['Season'] == season]
    n = sdata[BC['IR']].notna().sum()

    # Top: absolute BC
    ax = axes[0, idx]
    for name, info in WL.items():
        med = sdata.groupby('Hour')[BC[name]].median()
        ax.plot(med.index, med.values, 'o-', ms=3, lw=1.8, color=info['color'],
                label=f"{name} ({info['nm']})")
    ax.set_title(f'{season}\n(absolute BC)', fontweight='bold', color=SCOL[season])
    ax.set_xlabel('Hour')
    if idx == 0: ax.set_ylabel('BC (ug/m3)')
    ax.set_xticks(range(0, 24, 4))
    if idx == 2: ax.legend(fontsize=7, loc='upper right')
    ax.text(0.02, 0.98, f'n={n:,}', transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', fc='w', alpha=0.8))

    # Bottom: UV/IR ratio
    ax = axes[1, idx]
    for name in ['UV', 'Blue', 'Green', 'Red']:
        med = sdata.groupby('Hour')[ratio_cols[name]].median()
        ax.plot(med.index, med.values, 'o-', ms=3, lw=1.8, color=WL[name]['color'],
                label=f"{name}/IR")
    ax.axhline(1.0, color='gray', ls='--', lw=1)
    ax.set_title(f'{season}\n(spectral shape)', fontweight='bold', color=SCOL[season])
    ax.set_xlabel('Hour')
    if idx == 0: ax.set_ylabel('Wavelength / IR ratio')
    ax.set_xticks(range(0, 24, 4))
    if idx == 2: ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Diurnal Multi-Wavelength Patterns by Ethiopian Season', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT}/slide10_seasonal_wavelengths.png')
plt.show()

In [ ]:
# Slide 11: Representative Individual Days
daily_cv = df.groupby('Date')['wl_CV'].median().dropna()
daily_count = df.groupby('Date')[BC['IR']].count()
good_days = daily_count[daily_count >= 1200].index
daily_cv = daily_cv.loc[daily_cv.index.isin(good_days)]

high_cv_day = daily_cv.nlargest(20).index[5]
low_cv_day = daily_cv.nsmallest(20).index[5]
med_cv_day = daily_cv.iloc[(daily_cv - daily_cv.median()).abs().argsort()[:1]].index[0]

example_days = [
    (high_cv_day, 'High wavelength divergence'),
    (med_cv_day, 'Typical day'),
    (low_cv_day, 'Low divergence (uniform)'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for idx, (day, desc) in enumerate(example_days):
    ax = axes[idx]
    day_data = df[df['Date'] == day]
    for name, info in WL.items():
        col = BC_SM.get(name, BC[name])
        if col in day_data.columns:
            ax.plot(day_data.index, day_data[col], lw=1.2, color=info['color'],
                    label=f"{name} ({info['nm']} nm)", alpha=0.85)
    season = day_data['Season'].iloc[0] if len(day_data) > 0 else '?'
    ax.set_title(f'{day} -- {desc} (CV={daily_cv[day]:.2f}, {season})',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('BC (ug/m3)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    if idx == 0:
        ax.legend(fontsize=7, loc='upper right', ncol=5)
    if idx == 2:
        ax.set_xlabel('Time of Day (local)')

plt.suptitle('Representative Individual Days -- Multi-Wavelength BC', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT}/slide11_example_days.png')
plt.show()

In [ ]:
# Slide 12: Heatmap + normalized lines
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
wl_names = list(WL.keys())
heatmap_data = np.zeros((len(wl_names), 24))
for i, name in enumerate(wl_names):
    hourly_med = df.groupby('Hour')[BC[name]].median()
    daily_med = hourly_med.mean()
    heatmap_data[i, :] = hourly_med.values / daily_med if daily_med > 0 else 0

ax = axes[0]
wl_labels = [f"{name} ({WL[name]['nm']})" for name in wl_names]
im = ax.imshow(heatmap_data, aspect='auto', cmap='RdYlBu_r', vmin=0.4, vmax=1.8)
ax.set_yticks(range(len(wl_labels)))
ax.set_yticklabels(wl_labels)
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h:02d}' for h in range(0, 24, 2)])
ax.set_xlabel('Hour of Day')
ax.set_title('Diurnal Amplification by Wavelength\n(1.0 = daily average)')
plt.colorbar(im, ax=ax, label='Ratio to daily median')

ax = axes[1]
for i, name in enumerate(wl_names):
    info = WL[name]
    ax.plot(range(24), heatmap_data[i, :], 'o-', lw=2, ms=4,
            color=info['color'], label=f"{name} ({info['nm']})")
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Ratio to daily median')
ax.set_title('Diurnal Amplification Curves\n(lines close = similar; separated = divergent)')
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUT}/slide12_heatmap.png')
plt.show()

spread_by_hour = np.std(heatmap_data, axis=0)
print(f"Hours with most divergence: {np.argsort(spread_by_hour)[-3:][::-1].tolist()}")
print(f"Hours with most convergence: {np.argsort(spread_by_hour)[:3].tolist()}")

In [ ]:
# Slide 15: Formatted Summary Statistics Table
summary_rows = []
for season in list(SEASONS.keys()) + ['All']:
    sdata = df if season == 'All' else df[df['Season'] == season]
    for block_label, (h0, h1) in time_blocks.items():
        mask = (sdata['Hour'] >= h0) & (sdata['Hour'] < h1)
        block = sdata[mask]
        summary_rows.append({
            'Season': season if season != 'All' else 'All Seasons',
            'Time Block': block_label.replace('\n', ' '),
            'n': f"{block[BC['IR']].notna().sum():,}",
            'IR BCc': f"{block[BC['IR']].median():.1f}",
            'UV/IR': f"{block['UV/IR ratio'].median():.2f}" if 'UV/IR ratio' in block.columns else '',
            'True AAE': f"{block['AAE_true'].median():.2f}",
            'Delta-C': f"{block['Delta_C'].median():.1f}",
            'CV': f"{block['wl_CV'].median():.3f}",
        })

summary = pd.DataFrame(summary_rows)

# Render as figure-table for the slide
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

# Filter to "All Seasons" for the main table
all_data = summary[summary['Season'] == 'All Seasons']
cols = ['Time Block', 'n', 'IR BCc', 'UV/IR', 'True AAE', 'Delta-C', 'CV']
col_labels = ['Time Block', 'n (minutes)', 'IR BCc\n(ug/m3)', 'UV/IR\nratio', 'True\nAAE', 'Delta-C\n(ug/m3)', 'Wl CV']

table = ax.table(cellText=all_data[cols].values,
                 colLabels=col_labels,
                 cellLoc='center', loc='upper center',
                 colColours=['#B85042']*len(cols))
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

# Header styling
for j in range(len(cols)):
    table[0, j].set_text_props(color='white', fontweight='bold')
    table[0, j].set_facecolor('#B85042')

# Alternate row colors
for i in range(1, len(all_data) + 1):
    for j in range(len(cols)):
        table[i, j].set_facecolor('#F5E6D3' if i % 2 == 0 else 'white')

ax.set_title('Diurnal Wavelength Metrics by Time Block (All Seasons)\n'
             'True AAE > 1.0 at all hours = persistent biomass/brown carbon',
             fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(f'{OUT}/slide15_summary_table.png')
plt.show()

# Save full CSV too
summary.to_csv(f'{OUT}/diurnal_summary_full.csv', index=False)
print(f'Saved {OUT}/diurnal_summary_full.csv')